## Introduction
This notebook prepares data for modeling by engineering additional features, splitting the dataset into training, validation, and test sets, and applying training-based preprocessing transformations.

Close price is the target variable, and selected numerical and categorical variables are used as features. The training window length is tuned during validation.

In [1]:
import numpy as np
import pandas as pd

In [2]:
sold = pd.read_csv('clean-data/sold_clean.csv')
sold.shape

/var/folders/t2/p9112v_n469068__fty8_3nc0000gn/T/ipykernel_48885/394573382.py:1: DtypeWarning: Columns (39,40,41,42) have mixed types. Specify dtype option on import or set low_memory=False.
  sold = pd.read_csv('clean-data/sold_clean.csv')


(328866, 43)

## Part 1: Feature Engineering
#### 1.1 Property Age
Infer property age from built year.

In [3]:
sold['property_age'] = pd.to_datetime(sold['CloseDate']).dt.year - sold['YearBuilt']
sold['property_age'].describe()

count    328707.000000
mean         48.886145
std          27.420830
min           0.000000
25%          27.000000
50%          49.000000
75%          69.000000
max         249.000000
Name: property_age, dtype: float64

#### 1.2 Living Area per Bedroom

In [4]:
sold_with_beds = sold[sold['BedroomsTotal'] > 0]
sold['living_area_per_bedroom'] = sold_with_beds['LivingArea'] / sold_with_beds['BedroomsTotal']
sold['living_area_per_bedroom'].describe()

count    328723.000000
mean        577.990596
std         197.454453
min           1.000000
25%         445.333333
50%         539.333333
75%         662.500000
max        8050.000000
Name: living_area_per_bedroom, dtype: float64

#### 1.3 Bathroom-to-Bedroom Ratio

In [5]:
sold['bath_bed_ratio'] = sold_with_beds['BathroomsTotalInteger'] / sold_with_beds['BedroomsTotal']
sold['bath_bed_ratio'].describe()

count    328673.000000
mean          0.753432
std           0.294798
min           0.000000
25%           0.666667
50%           0.666667
75%           1.000000
max          87.500000
Name: bath_bed_ratio, dtype: float64

#### 1.4 Livable-to-Lot Space Ratio (i.e. Floor Area Ratio)

In [6]:
sold_with_lot = sold[sold['LotSizeSquareFeet'] > 0]
sold['floor_area_ratio'] = sold_with_lot['LivingArea'] / sold_with_lot['LotSizeSquareFeet']
sold['floor_area_ratio'].describe()

count    3.229920e+05
mean     1.353203e+01
std      6.249384e+02
min      6.365655e-07
25%      1.666419e-01
50%      2.359174e-01
75%      3.336013e-01
max      2.386000e+05
Name: floor_area_ratio, dtype: float64

#### 1.5 School District
Identify the school district for each property by spatially mapping school district polygons to each property's geographic coordinates.

In [7]:
import geopandas as gpd
from shapely.geometry import Point

In [8]:
sold_no_missing_coords = sold[sold['Longitude'].notna() & sold['Latitude'].notna()]
geometry = [Point(xy) for xy in zip(sold_no_missing_coords['Longitude'], sold_no_missing_coords['Latitude'])]
sold_gdf = gpd.GeoDataFrame(
    sold_no_missing_coords, 
    geometry=geometry,
    crs='EPSG:4326' # use the standard GPS system to transform coordinates
)
sold_gdf.shape

(317772, 48)

In [9]:
districts_gdf = gpd.read_file('raw-data/DistrictAreas2425/DistrictAreas2425.shp')

# unify coordinate systems before joining
if districts_gdf.crs != sold_gdf.crs:
    districts_gdf = districts_gdf.to_crs(sold_gdf.crs)

# NOTE: Calfornia school districts are grouped into three tiers (unifed, high, or elementary), and a property may reside simultaneously in multiple districts
# spatially join by district type to avoid introducing duplicates
high_schools = districts_gdf[districts_gdf['DistrictTy'] == 'High']
elementary_schools = districts_gdf[districts_gdf['DistrictTy'] == 'Elementary']
unified_schools = districts_gdf[districts_gdf['DistrictTy'] == 'Unified']

# map the polygons for each district type
# NOTE: the "within" predicate assigns a school district only if the property coordinates fall within the corresponding polygon
high_joined = gpd.sjoin(sold_gdf, high_schools, how='left', predicate='within')
elementary_joined = gpd.sjoin(sold_gdf, elementary_schools, how='left', predicate='within')
unified_joined = gpd.sjoin(sold_gdf, unified_schools, how='left', predicate='within')

In [10]:
district_types_combined = (
    high_joined[['DistrictNa', 'DistrictTy']]
    .fillna(elementary_joined[['DistrictNa', 'DistrictTy']])
    .fillna(unified_joined[['DistrictNa', 'DistrictTy']])
)

sold = sold.join(district_types_combined, how='left')
sold.shape

(328866, 49)

## Part 2: Chronological Split
Split the dataset chronologically into training, validation, and test sets. The most recent month is reserved for testing, the second most recent month for validation, and a variable-length historical window preceding the validation period for training. The length of the training window is tuned as a hyperparameter during model validation.

In [11]:
# hyperparameter to tune
months_preceding = 12

In [12]:
close_month = pd.to_datetime(sold['CloseDate']).dt.to_period('M')
test_month = close_month.max()
validation_month = close_month.max() - 1
train_start_month = validation_month - months_preceding

In [13]:
sold_test = sold[close_month == test_month]
sold_validation = sold[close_month == validation_month]
sold_train = sold[(close_month >= train_start_month) & (close_month < validation_month)]

In [14]:
splits = {
    'Split': ['Train', 'Validation', 'Test'],
    'Size': [sold_train.shape[0], sold_validation.shape[0], sold_test.shape[0]],
    'Start Date': [sold_train['CloseDate'].min(), sold_validation['CloseDate'].min(), sold_test['CloseDate'].min()],
    'End Date': [sold_train['CloseDate'].max(), sold_validation['CloseDate'].max(), sold_test['CloseDate'].max()]
}
pd.DataFrame(splits)

,Split,Size,Start Date,End Date
0,Train,128998,2025-05-01,2026-04-30
1,Validation,11836,2026-05-01,2026-05-31
2,Test,12656,2026-06-01,2026-06-30


## Part 3: Training-Based Transformations
Learn imputation, outlier thresholds, scaling, and encoding parameters from the training data, then apply the learned transformations to the validation and test sets.

Outlier thresholds are calculated below, while all other preprocessing transformations are implemented as functions in a standalone text file for use during model development.

#### Outlier Detection
Calculate the 0.5th and 99.5th percentiles of close price from the training data. Treat any record with a close price outside these bounds as an outlier and drop these records from the training, validation, and test sets.

In [15]:
y_train = sold_train['ClosePrice']
y_validation = sold_validation['ClosePrice']
y_test = sold_test['ClosePrice']

In [16]:
lower_price = y_train.quantile(0.005)
upper_price = y_train.quantile(0.995)
print(f'Lower price threshold: ${lower_price:,.0f}')
print(f'Upper price threshold: ${upper_price:,.0f}')

Lower price threshold: $195,000
Upper price threshold: $8,500,000


In [17]:
train_price_cleaned = y_train.between(lower_price, upper_price, inclusive='both')
validation_price_cleaned = y_validation.between(lower_price, upper_price, inclusive='both')
test_price_cleaned = y_test.between(lower_price, upper_price, inclusive='both')

In [18]:
sold_train = sold_train.loc[train_price_cleaned].reset_index(drop=True)
sold_validation = sold_validation.loc[validation_price_cleaned].reset_index(drop=True)
sold_test = sold_test.loc[test_price_cleaned].reset_index(drop=True)

In [19]:
splits = {
    'Split': ['Train', 'Validation', 'Test'],
    'Size Before': splits['Size'],
    'Size After': [sold_train.shape[0], sold_validation.shape[0], sold_test.shape[0]], 
    '% Removed': [(1 - sold_train.shape[0] / splits['Size'][0]) * 100, 
                  (1 - sold_validation.shape[0] / splits['Size'][1]) * 100, 
                  (1 - sold_test.shape[0] / splits['Size'][2]) * 100]
}
pd.DataFrame(splits)

,Split,Size Before,Size After,% Removed
0,Train,128998,127721,0.989938
1,Validation,11836,11722,0.963163
2,Test,12656,12523,1.050885


Further calculate the 0.5th and 99.5th percentiles of price per square foot from the training data to filter out data entry errors and rare sales.

In [20]:
price_per_sqft_train = sold_train['ClosePrice'] / sold_train['LivingArea']
price_per_sqft_validation = sold_validation['ClosePrice'] / sold_validation['LivingArea']
price_per_sqft_test = sold_test['ClosePrice'] / sold_test['LivingArea']

In [21]:
lower_price_per_sqft = price_per_sqft_train.quantile(0.005)
upper_price_per_sqft = price_per_sqft_train.quantile(0.995)
print(f'Lower price per sqft threshold: ${lower_price_per_sqft:,.0f}')
print(f'Upper price per sqft threshold: ${upper_price_per_sqft:,.0f}')

Lower price per sqft threshold: $163
Upper price per sqft threshold: $2,081


In [22]:
train_price_per_sqft_cleaned = price_per_sqft_train.between(lower_price_per_sqft, upper_price_per_sqft, inclusive='both')
validation_price_per_sqft_cleaned = price_per_sqft_validation.between(lower_price_per_sqft, upper_price_per_sqft, inclusive='both')
test_price_per_sqft_cleaned = price_per_sqft_test.between(lower_price_per_sqft, upper_price_per_sqft, inclusive='both')

In [23]:
sold_train = sold_train.loc[train_price_per_sqft_cleaned].reset_index(drop=True)
sold_validation = sold_validation.loc[validation_price_per_sqft_cleaned].reset_index(drop=True)
sold_test = sold_test.loc[test_price_per_sqft_cleaned].reset_index(drop=True)

In [24]:
splits = {
    'Split': ['Train', 'Validation', 'Test'],
    'Size Before': splits['Size After'],
    'Size After': [sold_train.shape[0], sold_validation.shape[0], sold_test.shape[0]], 
    '% Removed': [(1 - sold_train.shape[0] / splits['Size After'][0]) * 100, 
                  (1 - sold_validation.shape[0] / splits['Size After'][1]) * 100, 
                  (1 - sold_test.shape[0] / splits['Size After'][2]) * 100]
}
pd.DataFrame(splits)

,Split,Size Before,Size After,% Removed
0,Train,127721,126443,1.000619
1,Validation,11722,11588,1.143150
2,Test,12523,12385,1.101972


## Conclusion
School district, property age, and several spatial ratios were added as new features, and the dataset was split into training, validation, and test sets, with each set transformed using preprocessing parameters learned exclusively from the training data to ensure reliable model evaluation.

Export the transformed datasets for subsequent modeling.

In [25]:
sold_train.to_csv('clean-data/sold_train.csv', index=False)
sold_validation.to_csv('clean-data/sold_validation.csv', index=False)
sold_test.to_csv('clean-data/sold_test.csv', index=False)